<style>
  .cell-markdown { overflow: auto !important; }
  .mermaid { max-width: 100%; height: auto; }
</style>

# Session + Threshold Windows Walkthrough


## Overview

[Preparation](#prep)

* [Topology](#topology)
* [Steps](#steps)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [11]:
import sys
sys.path.insert(1, "../..")
sys.path.insert(1, "../../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import OrderGenerator
order_generator = OrderGenerator()

order_source_str = "orders"
sink_str = "sink"

#

def process(built_tn, customer_id, price, ts, w=1):
    m = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    #
    sink_str_r_list_dict = built_tn.process({order_source_str: [(m, w)]})
    r_list = sink_str_r_list_dict[sink_str]
    #
    print("Triggers:")
    for r in r_list:
        print(r)


<a id="topology"></a>
---
## Topology

Now it's time for the walkthrough itself. We go for a slightly simpler example for the walkthroughs.

The corresponding test can be found here: [test_windows.py](../../../../test/streams/test_windows.py)

In [20]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

gap_int = 20
max_session_int = 200
allowed_lateness_int = gap_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_session(lambda r: r["ts"], max_session_int, allowed_lateness_int)
    .distinct()
)
#
sink_tn = order_tn.group_by_agg_session(
    ts_fun=lambda r: r["ts"],
    gap_int=gap_int,
    key_fun=lambda r: r["customer_id"], 
    agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,           
                            "total_price": agg_r["total_price"] + r["price"],
                            "last_ts": max(agg_r["last_ts"], r["ts"])},
    agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0},
    project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                        "orders": agg_r["orders"],
                                        "total_price": agg_r["total_price"],
                                        "last_ts": agg_r["last_ts"]},
    trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1] or r_end_ts_tuple[0]["total_price"] > 200, 
    trigger_positive_only_bool=False
).sink(sink_str)
#
_ = built_tn = Tn.build(sink_tn)

What do we do?
1. `map()`: Select customer_id, price and ts from the value.
2. `expire_session()`: Expire with max seesion size `200` and allowed_lateness `gap_int = 20 * 3 = 60`
3. `distinct()`: Deduplicate.
4. `group_by_agg_tumbling()`: Set up the session/threshold window: group by customer ID, count the orders sum up the prices of the orders and get the last timestamp of the window.

To convert the ordinary session into a session/threshold window, we modify the default trigger function to also trigger if the total price is greater than `200`.

Next, we illustrate how the session/threshold window works by processing some example data - one by one, in baby steps.

We use two types of illustrations in each step:
1. Top-down view:
  * time proceeds from top to bottom (starting with `0`)
  * small grey circles mark the time every `100` ms for clarity
  * the latest timestamp of the input after the respective step is written at the top 
  * new events coming in a step are indicated a blue frame
  * old events have a grey frame
  * the triggered outputs in the sink are indicated by green color
2. Left-right view:
  * time proceeds from left to right (starting with `0`)
  * new windows appear below the old windows
  * `^`: latest timestamp of this step
  * `(^)`: latest timestamp of the previous step
  * `<<<`: time window(s) containing the event from this step
  * `!!!`: time window(s) triggered by the event from this step


<a id="steps"></a>
## Steps

### Step 1

In step 1, first, the first order arrives from customer 1 at timestamp `10`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 10"]
        direction TB
        0(("0")) e1@-.-> 10
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* advances the latest timestamp from `0` (start, visualized by `(^)`) to `10` (visualized by `^`),
* falls into the first session window `[10, 30)`, i.e., from `10` until `29` (visualized by `<<<`),
* and triggers nothing:
```
0 ---------- 100 ---------- 200 ---------- 300 ---------- 400
(^)    
  ^
[10 -- 30) (customer_id = 1) <<<
```

Session windows (in the "traditional" Kafka Streams sense and also in Kafi Streams) are, like sliding windows, indepenedent for each key, i.e., in this case, independent for each customer ID. Hence in this walkthrough, we also mark the time windows by their respective customer ID.

Let's see this happening for real:

In [21]:
process(built_tn, customer_id=1, price=100, ts=10, w=1)


Triggers:


### Step 2

Another order arrives from customer 1 at timestamp `25`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 25"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 25
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    25@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 25}"}
    style 25 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 300,\n&quot;last_ts&quot;: 25\n&quot;window_end&quot;: 45}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    25 --- space1
    style space1 fill:none,stroke:none
    linkStyle 2 stroke:none 
    25 Link@== Triggers ==> Output
    linkStyle 3 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `10` to `25`,
* falls into the window `[10, 45)` resulting from the merge of `[10, 30)` and `[25, 45)` since the gap between the two is `25 - 10 = 15 < gap_int = 20`,
* and *does* trigger window `[10, 45)` already since the threshold (`total_price = 300 > 200`) is already exceeded:
```
0 ---------- 100 ---------- 200 ---------- 300 ---------- 400
 (^)    
    ^
[10 ---- 45) (customer_id = 1) <<< merged from [10, 30) and [25, 45) !!!
```


In [22]:
process(built_tn, customer_id=1, price=200, ts=25, w=1)


Triggers:
{'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 25, 'window_end': 45}


### Step 3

Now an order from customer 2 arrives (at timestamp 75):

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 75"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 25 e3@-.-> 75
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    25@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 25}"}
    style 25 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* advances the latest timestamp from `25` to `75`,
* falls into the window `[75, 95)`,
* and does not trigger anything (window `[10, 45)` has already been triggered in the previous step):
```
0 ---------- 100 ---------- 200 ---------- 300 ---------- 400
   (^)    
         ^
[10 ---- 45) (customer_id = 1)
         [75 -- 95) (customer_id = 2) <<<
```


In [23]:
process(built_tn, customer_id=2, price=50, ts=75, w=1)

Triggers:


### Step 4

An order from customer 1 arrives out-of-order at timestamp `15`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 75"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 15 e3@-.-> 25 e4@-.-> 75
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    25@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 25}"}
    style 25 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 3,\n&quot;total_price&quot;: 350,\n&quot;last_ts&quot;: 25\n&quot;window_end&quot;: 45}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    15 Link@== Triggers ==> Output
    linkStyle 4 stroke:#00bb00,stroke-width:3px
```

This event:
* does not advance the latest timestamp (it stays at `75`),
* falls into the window `[10, 45)`,
* and triggers a correction of the window `[10, 45)`:
```
0 ---------- 100 ---------- 200 ---------- 300 ---------- 400
   (^)    
         ^
[10 ---- 45) (customer_id = 1) <<< !!!
         [75 -- 95) (customer_id = 2)
```


In [24]:
process(built_tn, customer_id=1, price=50, ts=15, w=1)


Triggers:
{'customer_id': 1, 'orders': 3, 'total_price': 350, 'last_ts': 25, 'window_end': 45}


### Step 5

Yet another order from customer 1 arrives at timestamp `220`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 220"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 15 e3@-.-> 25 e4@-.-> 75 e5@-.-> 100(("100")) e6@-.-> 200(("200")) e7@-.-> 220
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    25@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 25}"}
    style 25 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#333,stroke-width:6px

    220@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 500,\n&quot;ts&quot;: 220}"}
    style 220 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50,\n&quot;last_ts&quot;: 75\n&quot;window_end&quot;: 95}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    220 Link@== Triggers ==> Output1
    linkStyle 7 stroke:#00bb00,stroke-width:3px

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 500,\n&quot;last_ts&quot;: 220\n&quot;window_end&quot;: 240}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    220 Link@== Triggers ==> Output2
    linkStyle 8 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `75` to `220`,
* falls into the window `[220, 240)`,
* and triggers the window `[75, 95)`*and* the window `[220, 240)` a well since the total price threshold of `200` is already exceed with this one event (`price` = `500`):
```
0 ---------- 100 ---------- 200 ---------- 300 ---------- 400
        (^)    
                                 ^
[10 ---- 45) (customer_id = 1)
         [75 -- 95) (customer_id = 2) !!!
                                 [220 -- 240] (customer_id = 1) <<< !!!
```


In [25]:
process(built_tn, customer_id=1, price=500, ts=220, w=1)

Triggers:
{'customer_id': 1, 'orders': 1, 'total_price': 500, 'last_ts': 220, 'window_end': 240}
{'customer_id': 2, 'orders': 1, 'total_price': 50, 'last_ts': 75, 'window_end': 95}


### Step 6

Another order from customer 1 arrives out-of-order at timestamp `1`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 220"]
        direction TB
        0(("0")) e1@-.-> 1 e2@-.-> 10 e3@-.-> 15 e4@-.-> 25 e5@-.-> 75 e6@-.-> 100(("100")) e7@-.-> 200(("200")) e8@-.-> 220
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    25@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 25}"}
    style 25 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#333,stroke-width:6px

    220@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 500,\n&quot;ts&quot;: 220}"}
    style 220 fill:none,stroke:#333,stroke-width:6px

    1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 999,\n&quot;ts&quot;: 1}"}
    style 1 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 4,\n&quot;total_price&quot;: 1349,\n&quot;last_ts&quot;: 25\n&quot;window_end&quot;: 45}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    1 Link@== Triggers ==> Output
    linkStyle 8 stroke:#00bb00,stroke-width:3px
```

This event:
* does not advance the latest timestamp (it stays at `220`),
* falls into the window `[1, 45)`, the result of merging `[1, 21)` and `[10, 45)`,  
* and triggers a correction of `[1, 45)`:
```
0 ---------- 100 ---------- 200 ---------- 300 ---------- 400
                                (^)    
                                 ^
[1 ---- 45) (customer_id = 1) <<< merged from [1, 21) and [10, 45) !!!
         [75 -- 95) (customer_id = 2)
                                 [220 -- 240] (customer_id = 1)
```


In [26]:
process(built_tn, customer_id=1, price=999, ts=1, w=1)


Triggers:
{'customer_id': 1, 'orders': 4, 'total_price': 1349, 'last_ts': 25, 'window_end': 45}


### Step 7

An order from customer 1 arrives at timestamp `330`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 330"]
        direction TB
        0(("0")) e1@-.-> 1 e2@-.-> 10 e3@-.-> 15 e4@-.-> 25 e5@-.-> 75 e6@-.-> 100(("100")) e7@-.-> 200(("200")) e8@-.-> 220 e9@-.-> 300(("300")) e10@-.-> 330
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    25@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 25}"}
    style 25 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#333,stroke-width:6px

    220@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 500,\n&quot;ts&quot;: 220}"}
    style 220 fill:none,stroke:#333,stroke-width:6px

    1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 999,\n&quot;ts&quot;: 1}"}
    style 1 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* advances the latest timestamp from `220` to `330`,
* falls into the window `[330, 350)`,  
* and triggers nothing:
```
0 ---------- 100 ---------- 200 ---------- 300 ---------- 400
                                (^)    
                                                 ^
[1 ---- 45) (customer_id = 1)
         [75 -- 95) (customer_id = 2)
                                 [220 -- 240] (customer_id = 1)
                                                 [330 -- 350] (customer_id = 1) <<<
```


In [27]:
process(built_tn, customer_id=1, price=100, ts=330, w=1)


Triggers:


### Step 8

An order from customer 2 arrives too late at timestamp `2`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 330"]
        direction TB
        0(("0")) e1@-.-> 1 e2@-.-> 2 e3@-.-> 10 e4@-.-> 15 e5@-.-> 25 e6@-.-> 75 e7@-.-> 100(("100")) e8@-.-> 200(("200")) e9@-.-> 220 e10@-.-> 300(("300")) e11@-.-> 330
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }
    e11@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    25@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 25}"}
    style 25 fill:none,stroke:#333,stroke-width:6px

    75@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 75}"}
    style 75 fill:none,stroke:#333,stroke-width:6px

    15@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 15}"}
    style 15 fill:none,stroke:#333,stroke-width:6px

    220@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 500,\n&quot;ts&quot;: 220}"}
    style 220 fill:none,stroke:#333,stroke-width:6px

    1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 999,\n&quot;ts&quot;: 1}"}
    style 1 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#333,stroke-width:6px

    2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 2}"}
    style 2 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* does not advance the latest timestamp (it stays at `330`),
* falls into the window `[2, 22)`,  
* but it is too late (`allowed_lateness = 200`) to trigger anything: `330 (latest) - ( 21 (window end for ts = 1) ) = 309 > 200`:
```
0 ---------- 100 ---------- 200 ---------- 300 ---------- 400
                                                (^)    
                                                 ^
[1 ---- 45) (customer_id = 1)
 (2 -- 22] (customer_id = 2) <<<
         [75 -- 95) (customer_id = 2)
                                 [220 -- 240] (customer_id = 1)
                                                 [330 -- 350] (customer_id = 1)
```


In [28]:
process(built_tn, customer_id=2, price=200, ts=2, w=1)


Triggers:
